# RAG Chat — OPD Kab Batang

Eksperimen retrieval & generation untuk direktori OPD (61 records). Vector store sudah ter-persist dari `build_vectorstore_opd.ipynb`.

**Tiga retrieval variants** (reranker dilewat karena 61 docs sangat kecil):
- **V1 — Dense**: similarity search top-k
- **V2 — BM25**: pure keyword (cocok untuk nama OPD exact)
- **V3 — Hybrid**: BM25 + Dense (RRF via EnsembleRetriever)

## Step 1 — Load Vector Store

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert api_key, "GEMINI_API_KEY tidak ditemukan di .env"
os.environ["GOOGLE_API_KEY"] = api_key
print("API key OK")

In [ ]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# PENTING: task_type="retrieval_query" untuk embedding query (beda dengan build_vectorstore_opd)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_query",
    output_dimensionality=768,
)

vectorstore = Chroma(
    collection_name="opd_directory",
    embedding_function=embeddings,
    persist_directory="../data/opd_vector_store",
)

n = vectorstore._collection.count()
print(f"Loaded {n} OPD docs dari vector store")
assert n > 0, "Vector store kosong — jalankan build_vectorstore_opd.ipynb dulu!"

In [ ]:
# Load semua docs ke memori untuk BM25 (V2 & V3)
from langchain_core.documents import Document

raw = vectorstore.get()
all_docs = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(raw['documents'], raw['metadatas'])
]
print(f"Loaded {len(all_docs)} docs in-memory untuk BM25")

## Step 2 — Setup LLM + Prompt Template

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_tokens=1024,
)

test = llm.invoke("Jawab dengan satu kata: ibu kota Indonesia?")
print(f"LLM OK. Response: {test.content}")

In [ ]:
PROMPT_TEMPLATE = """Kamu adalah asisten direktori OPD (Organisasi Perangkat Daerah) Kabupaten Batang.

ATURAN:
1. Jawab HANYA berdasarkan konteks di bawah. Jangan menambah informasi dari pengetahuan umum.
2. Kalau OPD yang ditanya tidak ada dalam konteks, jawab: \"OPD tidak ditemukan dalam direktori.\"
3. Sebutkan sumber di akhir jawaban dalam format: [Sumber: <nama_opd>, nomor <nomor>]
4. Kalau pertanyaan menanyakan multiple OPD (mis. \"dinas apa saja\"), sebutkan semua yang relevan dari konteks.
5. Jawab dalam Bahasa Indonesia yang jelas dan ringkas.

KONTEKS:
{context}

PERTANYAAN: {question}

JAWABAN:"""

## Step 3 — Tiga Retrieval Variants

### Variant 1 — Dense (baseline)

In [ ]:
def retrieve_v1(query, k=5):
    return vectorstore.similarity_search(query, k=k)

# Test
results = retrieve_v1("alamat dinas pariwisata", k=3)
for r in results:
    m = r.metadata
    print(f"- [{m['nomor']}] {m['nama_opd']} ({m['tipe']})")

### Variant 2 — BM25 (keyword exact)

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(all_docs)
bm25.k = 5

def retrieve_v2(query, k=5):
    bm25.k = k
    return bm25.invoke(query)

# Test
results = retrieve_v2("alamat dinas pariwisata", k=3)
for r in results:
    m = r.metadata
    print(f"- [{m['nomor']}] {m['nama_opd']} ({m['tipe']})")

### Variant 3 — Hybrid (BM25 + Dense, RRF)

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
bm25_for_ens = BM25Retriever.from_documents(all_docs)
bm25_for_ens.k = 10

ensemble = EnsembleRetriever(
    retrievers=[bm25_for_ens, dense_retriever],
    weights=[0.5, 0.5],
)

def retrieve_v3(query, k=5):
    return ensemble.invoke(query)[:k]

# Test
results = retrieve_v3("alamat dinas pariwisata", k=3)
for r in results:
    m = r.metadata
    print(f"- [{m['nomor']}] {m['nama_opd']} ({m['tipe']})")

## Step 4 — `chat()` Function

In [ ]:
def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parent = f" (bagian dari {m['parent_opd']})" if m.get('parent_opd') else ""
        header = f"[Sumber {i}: {m.get('nama_opd', '?')}{parent}, nomor {m.get('nomor', '?')}, tipe {m.get('tipe', '?')}]"
        parts.append(f"{header}\n{d.page_content}")
    return "\n\n---\n\n".join(parts)


def chat(query, retriever_fn=None, k=5, show_sources=True, verbose=False):
    if retriever_fn is None:
        retriever_fn = retrieve_v3  # default ke hybrid
    docs = retriever_fn(query, k=k)
    context = format_context(docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=query)
    if verbose:
        print("=== PROMPT ===")
        print(prompt[:2000])
        print("...\n")
    answer = llm.invoke(prompt).content
    result = {"answer": answer}
    if show_sources:
        result["sources"] = [
            {
                "nomor": d.metadata.get('nomor'),
                "nama_opd": d.metadata.get('nama_opd'),
                "parent_opd": d.metadata.get('parent_opd'),
                "tipe": d.metadata.get('tipe'),
            }
            for d in docs
        ]
    return result

# Smoke test
result = chat("Alamat dan nomor telp Sekretariat Daerah", retriever_fn=retrieve_v3, k=5)
print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s}")

## Step 5 — Test Harness: Compare 3 Variants

Sample queries × 3 variants → bandingkan kualitas retrieval.

**Catatan**: query `\"Nomor telp Disdukcapil\"` adalah paraphrase test — nama lengkap di buku adalah \"Dinas Kependudukan dan Pencatatan Sipil\". Expect V1 (dense) & V3 (hybrid) bisa nyangkut, V2 (BM25 only) mungkin miss karena keyword exact ga match.

In [ ]:
TEST_QUERIES = [
    "Alamat dan nomor telp Sekretariat Daerah",
    "Email Bagian Kesejahteraan Rakyat",
    "Dinas apa saja yang ada di Kab Batang?",
    "Kecamatan Pecalungan alamatnya dimana?",
    "Apakah ada RSUD di Batang?",
    "Nomor telp Disdukcapil",  # paraphrase test
]

VARIANTS = {
    "V1 (Dense)":  retrieve_v1,
    "V2 (BM25)":   retrieve_v2,
    "V3 (Hybrid)": retrieve_v3,
}

### Step 5A — Retrieval Inspection (tanpa LLM)

Lihat chunk apa yang diretrieve tiap variant. Tidak ada LLM call → aman dijalankan berkali-kali.

In [ ]:
def inspect_retrieval(query, retriever_fn, name, k=5):
    docs = retriever_fn(query, k=k)
    print(f"\n  [{name}] {len(docs)} docs")
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parent = f" ← {m['parent_opd']}" if m.get('parent_opd') else ""
        print(f"    [{i}] [{m['nomor']}] {m['nama_opd']} ({m['tipe']}){parent}")


for qi, query in enumerate(TEST_QUERIES):
    print("\n" + "=" * 100)
    print(f"QUERY [{qi+1}/{len(TEST_QUERIES)}]: {query}")
    for name, fn in VARIANTS.items():
        inspect_retrieval(query, fn, name, k=5)

### Step 5B — Full Chat (dengan LLM)

**Hati-hati**: 6 query × 3 variant = 18 Gemini API call. Ada sleep 10s antar query untuk hindari rate limit.

In [ ]:
import time

for qi, query in enumerate(TEST_QUERIES):
    print("\n" + "=" * 100)
    print(f"QUERY [{qi+1}/{len(TEST_QUERIES)}]: {query}")
    for name, fn in VARIANTS.items():
        try:
            r = chat(query, retriever_fn=fn, k=5)
            src_str = ", ".join(f"{s['nomor']}/{s['nama_opd'][:20]}" for s in r['sources'])
            print(f"\n  [{name}]")
            print(f"  sources: {src_str}")
            print(f"  answer : {r['answer'][:400]}{'…' if len(r['answer']) > 400 else ''}")
        except Exception as e:
            print(f"\n  [{name}] ERROR: {e}")
    if qi < len(TEST_QUERIES) - 1:
        print(f"\n  [rate-limit guard] sleeping 10s…")
        time.sleep(10)

## Step 6 — Interactive Single-Query Cell

Edit `MY_QUERY` dan `MY_VARIANT` untuk eksperimen cepat.

In [ ]:
MY_QUERY = "Alamat Disdukcapil dan nomor telponnya"
MY_VARIANT = retrieve_v3  # ganti ke retrieve_v1 / v2 / v3

result = chat(MY_QUERY, retriever_fn=MY_VARIANT, k=5, verbose=False)
print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s}")